# Modeling + P + KNN + Florian Samain

Notebook exemple conforme aux consignes du travail.  
Fichier conseillé : **`Titanic_model_P_KNN.ipynb`**

## 1. Objectif

Dans ce notebook, on entraîne un modèle **KNN** sur le dataset Titanic en utilisant une **sélection personnelle de variables (`P`)**.

La structure du notebook est pensée pour être **réutilisée facilement** avec les autres combinaisons :
- `P + KNN`
- `P + Decision Tree`
- `P + Random Forest`
- `VB + KNN`
- `VB + Decision Tree`
- `VT + Random Forest`

Il suffit ensuite de :
1. changer le **titre**,
2. changer le **nom du fichier**,
3. adapter la **sélection de variables**,
4. remplacer le **modèle** et la **grille d’hyperparamètres**.

## 2. Indicateur de performance choisi

**L’indicateur principal choisi est le F1-score.**

**Pourquoi ?**  
Le problème Titanic est une **classification binaire** (`survived` ou non) et il peut y avoir un **déséquilibre entre les classes**.  
Le **F1-score** est pertinent car il combine :
- la **précision** : parmi les prédictions positives, combien sont correctes ;
- le **rappel** : parmi les vrais positifs, combien ont été retrouvés.

Ainsi, le F1-score permet une évaluation plus équilibrée qu’une simple accuracy, surtout si une classe est moins représentée.

## 3. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV,
    cross_validate,
    learning_curve
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

# Affichage plus lisible
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

## 4. Chargement des données
> Adapte le chemin si ton fichier n'est pas au même endroit.

In [ ]:
# Exemple de chemin
df = pd.read_csv("../data/Titanic Dataset_clean.csv")

print("Dimensions du dataset :", df.shape)
display(df.head())

## 5. Définition de la cible et de la sélection de variables

Ici on utilise une **sélection personnelle (`P`)**.  
Tu peux modifier la liste `selected_features` selon ta combinaison.

In [ ]:
target = "survived"

# =========================
# Sélection personnelle (P)
# =========================
selected_features = [
    "pclass",
    "sex",
    "age",
    "sibsp",
    "parch",
    "fare",
    "embarked",
    "alone"
]

# On garde uniquement les colonnes présentes dans le dataset
selected_features = [col for col in selected_features if col in df.columns]

X = df[selected_features].copy()
y = df[target].copy()

print("Variables utilisées :", selected_features)
print("Shape X :", X.shape)
print("Répartition de y :")
print(y.value_counts(normalize=True))

## 6. Prétraitement minimal

Le KNN est un modèle basé sur les distances.  
Il est donc **important de standardiser les variables numériques**.

Dans cet exemple :
- on convertit les variables catégorielles avec `get_dummies`,
- on gère les valeurs manquantes avec `SimpleImputer`,
- on standardise avec `StandardScaler`.

In [ ]:
# Encodage simple des variables catégorielles
X_encoded = pd.get_dummies(X, drop_first=True)

print("Shape après encodage :", X_encoded.shape)
display(X_encoded.head())

## 7. Séparation train / test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train :", X_train.shape, y_train.shape)
print("Test  :", X_test.shape, y_test.shape)

## 8. Pipeline KNN

On utilise un pipeline pour enchaîner :
1. l'imputation,
2. la standardisation,
3. le modèle KNN.

In [ ]:
pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier())
])

pipeline

## 9. Optimisation des hyperparamètres

On optimise ici :
- `n_neighbors`
- `weights`
- `metric`

L'optimisation se fait avec :
- une **cross validation stratifiée**
- un **GridSearchCV**
- le **F1-score** comme critère principal.

In [ ]:
param_grid = {
    "model__n_neighbors": [3, 5, 7, 9, 11, 15],
    "model__weights": ["uniform", "distance"],
    "model__metric": ["euclidean", "manhattan", "minkowski"]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)

grid_search.fit(X_train, y_train)

print("Meilleurs hyperparamètres :", grid_search.best_params_)
print("Meilleur score CV (F1)    :", round(grid_search.best_score_, 4))

## 10. Résultats détaillés de la cross validation

In [ ]:
cv_results = pd.DataFrame(grid_search.cv_results_)

cols_to_show = [
    "params",
    "mean_train_score",
    "std_train_score",
    "mean_test_score",
    "std_test_score",
    "rank_test_score"
]

display(cv_results[cols_to_show].sort_values("rank_test_score").head(10))

## 11. Interprétation de la cross validation

À interpréter dans ton rapport :
- si le **score train** est beaucoup plus élevé que le **score validation**, cela peut indiquer un **surapprentissage** ;
- si les deux scores sont faibles, cela peut indiquer un **sous-apprentissage** ;
- si les scores train et validation sont proches et corrects, le modèle généralise mieux.

Le meilleur modèle sélectionné est celui qui maximise le **F1-score moyen en validation croisée**.

## 12. Entraînement du meilleur modèle

In [ ]:
best_model = grid_search.best_estimator_
best_model.fit(X_train, y_train)

## 13. Évaluation sur le jeu de test

In [ ]:
y_pred = best_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print("=== Scores sur le jeu de test ===")
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")

if hasattr(best_model, "predict_proba"):
    y_proba = best_model.predict_proba(X_test)[:, 1]
    roc_auc = roc_auc_score(y_test, y_proba)
    print(f"ROC-AUC   : {roc_auc:.4f}")
else:
    y_proba = None

## 14. Interprétation des indicateurs

Voici comment interpréter les résultats :

- **Accuracy** : proportion globale de bonnes prédictions.
- **Precision** : parmi les passagers prédits comme survivants, combien ont réellement survécu.
- **Recall** : parmi les vrais survivants, combien ont été retrouvés par le modèle.
- **F1-score** : compromis entre précision et rappel.
- **ROC-AUC** : capacité du modèle à séparer les deux classes sur différents seuils.

Dans l'analyse finale, il faut comparer ces indicateurs et expliquer si le modèle :
- privilégie la précision,
- privilégie le rappel,
- ou reste équilibré.

## 15. Rapport de classification

In [ ]:
print(classification_report(y_test, y_pred))

## 16. Matrice de confusion

In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.title("Matrice de confusion - KNN")
plt.show()

## 17. Courbe ROC

In [ ]:
if y_proba is not None:
    RocCurveDisplay.from_predictions(y_test, y_proba)
    plt.title("Courbe ROC - KNN")
    plt.show()
else:
    print("Le modèle ne fournit pas de probabilités.")

## 18. Analyse overfitting / underfitting avec learning curves

Les learning curves permettent de comparer :
- le score sur le **train**
- le score en **validation**

Interprétation :
- **train élevé / validation faible** → overfitting ;
- **train faible / validation faible** → underfitting ;
- **train proche de validation avec bons scores** → bon compromis.

In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    estimator=best_model,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring="f1",
    train_sizes=np.linspace(0.1, 1.0, 8),
    n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
val_mean = val_scores.mean(axis=1)
val_std = val_scores.std(axis=1)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_mean, label="Train score")
plt.plot(train_sizes, val_mean, label="Validation score")
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.2)
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.2)
plt.xlabel("Taille du jeu d'entraînement")
plt.ylabel("F1-score")
plt.title("Learning curve - KNN")
plt.legend()
plt.grid(True)
plt.show()

## 19. Cross validation multi-métriques

Cette cellule permet de montrer une analyse plus complète avec plusieurs métriques.

In [ ]:
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv_multi = cross_validate(
    estimator=best_model,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

results_multi = pd.DataFrame(cv_multi)
display(results_multi)

print("=== Moyennes CV ===")
for metric in ["test_accuracy", "test_precision", "test_recall", "test_f1", "test_roc_auc"]:
    print(f"{metric} : {results_multi[metric].mean():.4f} ± {results_multi[metric].std():.4f}")

## 20. Conclusion interprétée

Exemple de conclusion à adapter selon tes vrais résultats :

- Le modèle **KNN** a été optimisé par **GridSearchCV** avec validation croisée.
- La métrique principale choisie est le **F1-score**, car elle équilibre la précision et le rappel.
- Les résultats sur le jeu de test permettent de vérifier si le modèle généralise correctement.
- La comparaison entre les scores d'entraînement et de validation permet de détecter un éventuel **overfitting** ou **underfitting**.
- Si les learning curves montrent des scores proches entre train et validation, alors le modèle reste relativement stable.
- Si l'écart est important, cela signifie que le modèle doit être ajusté ou que la sélection de variables doit être améliorée.

**Conclusion générale :**  
Le KNN peut constituer une bonne base, mais ses performances dépendent fortement :
- de la qualité de la sélection de variables,
- de la standardisation,
- du choix de `k`,
- et de la métrique de distance.

## 21. Comment réutiliser ce notebook pour les autres modèles

### Pour Decision Tree
Changer :
- `KNeighborsClassifier()` par `DecisionTreeClassifier(random_state=42)`
- la grille d'hyperparamètres par exemple :
  - `max_depth`
  - `min_samples_split`
  - `min_samples_leaf`
  - `criterion`

### Pour Random Forest
Changer :
- `KNeighborsClassifier()` par `RandomForestClassifier(random_state=42)`
- la grille d'hyperparamètres par exemple :
  - `n_estimators`
  - `max_depth`
  - `min_samples_split`
  - `min_samples_leaf`

### Pour les autres sélections de variables
Modifier uniquement :
- `selected_features`
- le titre
- le nom du fichier